## 📚 Spark Union Operations - Learning Notes

### What This Notebook Is About
This notebook explores different ways to combine DataFrames in Spark, specifically focusing on **union operations**. It's basically about stacking DataFrames on top of each other (like appending rows).

### Key Concepts Covered

**1. union() vs unionAll() in PySpark**
* Both do the same thing in modern Spark - they keep ALL rows (including duplicates)
* `unionAll()` is deprecated but still works
* If you want to remove duplicates, use `.union().distinct()` or SQL `UNION`

**2. SQL UNION vs UNION ALL**
* `UNION` - Removes duplicate rows (slower because it needs to check)
* `UNION ALL` - Keeps all rows including duplicates (faster)

**3. Column Matching Rules**
* `.union()` matches columns **by position** (first column with first column, etc.)
* If columns are in different order, it will try to cast types and can fail!
* Column names don't matter for `.union()` - only the order and types

**4. unionByName() - The Safe Option**
* Matches columns **by name** instead of position
* Much safer when schemas might be in different orders
* Use `allowMissingColumns=True` if DataFrames have different numbers of columns

### Important Databricks/Spark Concepts

* **Schema Matching**: Spark needs compatible schemas to union DataFrames
* **Type Casting**: When column types don't match at the same position, Spark tries to cast them (can fail)
* **Temporary Views**: You can create SQL views from DataFrames to use SQL syntax
* **DataFrame Immutability**: Union creates a new DataFrame, doesn't modify the originals

### Things to Remember

✅ **Always check your schemas before union** - Use `.printSchema()` or `.show()` to verify column order

✅ **Use `unionByName()` when column order might differ** - It's safer and more explicit

✅ **PySpark `.union()` keeps duplicates** - Don't assume it removes them like SQL UNION

✅ **Column count must match for `.union()`** - Either select matching columns or use `unionByName()` with `allowMissingColumns=True`

⚠️ **Position-based union can silently give wrong results** - If you union DataFrames with columns in different order, you might get data in wrong columns!

### Quick Reference

```python
# Keep all rows including duplicates (by position)
df1.union(df2)

# Keep all rows, match by column name (safer)
df1.unionByName(df2)

# Handle different column counts
df1.unionByName(df2, allowMissingColumns=True)

# Remove duplicates after union
df1.union(df2).distinct()
```

```sql
-- Remove duplicates
SELECT * FROM table1 UNION SELECT * FROM table2

-- Keep duplicates (faster)
SELECT * FROM table1 UNION ALL SELECT * FROM table2
```

In [0]:
data=[(10 ,'Anil',50000, 18),
(11 ,'Vikas',75000,  16),
(12 ,'Nisha',40000,  18),
(13 ,'Nidhi',60000,  17),
(14 ,'Priya',80000,  18),
(15 ,'Mohit',45000,  18),
(16 ,'Rajesh',90000, 10),
(17 ,'Raman',55000, 16),
(18 ,'Sam',65000,   17),
(18 ,'Sam',65000,   17),
(18 ,'Sam',65000,   17)]

my_schema = ["id","name","salary","mngr_id"]

my_df = spark.createDataFrame(data , my_schema)

### Checking Row Count

We have 11 total rows including the 3 duplicate Sam entries. This baseline count will help us understand what happens when we union DataFrames.

### Creating a Second DataFrame

This is another employee DataFrame with 2 new employees (Sohan and Sima). Same schema as `my_df` - same column names in the same order.

### Using PySpark .union()

**Important:** In PySpark, `.union()` keeps ALL rows including duplicates. This is different from SQL UNION which removes duplicates.

We're combining `my_df` (11 rows) with `my_df1` (2 rows), so we get 13 total rows.

The name `.union()` is a bit misleading - it actually behaves like SQL's `UNION ALL`.

### Using .unionAll() (Deprecated)

`.unionAll()` does exactly the same thing as `.union()` - keeps all rows including duplicates.

This method is deprecated in modern Spark, so just use `.union()` instead. They both give the same result: 13 rows.

### Creating Temporary SQL Views

To demonstrate SQL union syntax, we're registering our DataFrames as temporary views. This lets us query them using SQL syntax in the next cells.

These views only exist in this session and aren't saved to the catalog.

### SQL UNION (Removes Duplicates)

**This is where it gets interesting!** SQL `UNION` (without ALL) removes duplicate rows.

We started with:
* `my_df`: 11 rows (including 3 duplicate Sams)
* `my_df1`: 2 rows
* Total before deduplication: 13 rows

After `UNION` removes duplicates, we get **10 rows** - the 3 duplicate Sam entries get reduced to 1.

This is the key difference: SQL `UNION` = deduplicated, PySpark `.union()` = keeps duplicates.

### SQL UNION ALL (Keeps Duplicates)

`UNION ALL` keeps all rows including duplicates, just like PySpark's `.union()` method.

Result: 13 rows (same as PySpark union)

**When to use which:**
* Use `UNION ALL` when you know there are no duplicates or you want to keep them (faster)
* Use `UNION` when you need to remove duplicates (slower because it has to check)

### SQL UNION on the Same Table

Here we're unioning `my_df` with itself. Without the `ALL` keyword, SQL UNION removes duplicates.

* Before: 11 rows + 11 rows = 22 rows total
* After deduplication: **8 rows**

This proves that `UNION` is deduplicating across the entire result set - it found only 8 unique employee records in the original 11 rows.

In [0]:
my_df.count()

11

In [0]:
data1=[(19 ,'Sohan',50000, 18),
(20 ,'Sima',75000,  17)]

my_schema = ["id","name","salary","mngr_id"]

my_df1 = spark.createDataFrame(data1 , my_schema)

In [0]:
my_df.union(my_df1).show()
my_df.union(my_df1).count()

+---+------+------+-------+
| id|  name|salary|mngr_id|
+---+------+------+-------+
| 10|  Anil| 50000|     18|
| 11| Vikas| 75000|     16|
| 12| Nisha| 40000|     18|
| 13| Nidhi| 60000|     17|
| 14| Priya| 80000|     18|
| 15| Mohit| 45000|     18|
| 16|Rajesh| 90000|     10|
| 17| Raman| 55000|     16|
| 18|   Sam| 65000|     17|
| 18|   Sam| 65000|     17|
| 18|   Sam| 65000|     17|
| 19| Sohan| 50000|     18|
| 20|  Sima| 75000|     17|
+---+------+------+-------+



13

In [0]:
my_df.unionAll(my_df1).show()
my_df.unionAll(my_df1).count()

+---+------+------+-------+
| id|  name|salary|mngr_id|
+---+------+------+-------+
| 10|  Anil| 50000|     18|
| 11| Vikas| 75000|     16|
| 12| Nisha| 40000|     18|
| 13| Nidhi| 60000|     17|
| 14| Priya| 80000|     18|
| 15| Mohit| 45000|     18|
| 16|Rajesh| 90000|     10|
| 17| Raman| 55000|     16|
| 18|   Sam| 65000|     17|
| 18|   Sam| 65000|     17|
| 18|   Sam| 65000|     17|
| 19| Sohan| 50000|     18|
| 20|  Sima| 75000|     17|
+---+------+------+-------+



13

In [0]:
my_df.createOrReplaceTempView("my_df")
my_df1.createOrReplaceTempView("my_df1")

In [0]:
spark.sql(""" select * from my_df union select * from my_df1""").show()
spark.sql(""" select * from my_df union select * from my_df1""").count()

+---+------+------+-------+
| id|  name|salary|mngr_id|
+---+------+------+-------+
| 10|  Anil| 50000|     18|
| 11| Vikas| 75000|     16|
| 12| Nisha| 40000|     18|
| 13| Nidhi| 60000|     17|
| 14| Priya| 80000|     18|
| 15| Mohit| 45000|     18|
| 16|Rajesh| 90000|     10|
| 17| Raman| 55000|     16|
| 18|   Sam| 65000|     17|
| 19| Sohan| 50000|     18|
| 20|  Sima| 75000|     17|
+---+------+------+-------+



11

In [0]:
spark.sql(""" select * from my_df union all select * from my_df1""").show()
spark.sql(""" select * from my_df union all select * from my_df1""").count()

+---+------+------+-------+
| id|  name|salary|mngr_id|
+---+------+------+-------+
| 10|  Anil| 50000|     18|
| 11| Vikas| 75000|     16|
| 12| Nisha| 40000|     18|
| 13| Nidhi| 60000|     17|
| 14| Priya| 80000|     18|
| 15| Mohit| 45000|     18|
| 16|Rajesh| 90000|     10|
| 17| Raman| 55000|     16|
| 18|   Sam| 65000|     17|
| 18|   Sam| 65000|     17|
| 18|   Sam| 65000|     17|
| 19| Sohan| 50000|     18|
| 20|  Sima| 75000|     17|
+---+------+------+-------+



13

In [0]:
spark.sql(""" select * from my_df union  select * from my_df""").show()
spark.sql(""" select * from my_df union  select * from my_df""").count()

+---+------+------+-------+
| id|  name|salary|mngr_id|
+---+------+------+-------+
| 10|  Anil| 50000|     18|
| 11| Vikas| 75000|     16|
| 12| Nisha| 40000|     18|
| 13| Nidhi| 60000|     17|
| 14| Priya| 80000|     18|
| 15| Mohit| 45000|     18|
| 16|Rajesh| 90000|     10|
| 17| Raman| 55000|     16|
| 18|   Sam| 65000|     17|
+---+------+------+-------+



9

In [0]:
#if column placing are shuffled then 
wrong_column_data=[(19 ,50000, 18,'Sohan'),
(20 ,75000,  17,'Sima')]

w_schema = ["id","salary","mngr_id","name"]

wrong_df = spark.createDataFrame(wrong_column_data, w_schema)

### ⚠️ Problem 1: What Happens When Column Order is Different?

Here we're creating a DataFrame with the **same columns but in a different order**:
* `my_df1` has: `["id", "name", "salary", "mngr_id"]`
* `wrong_df` has: `["id", "salary", "mngr_id", "name"]`

Notice how `name` moved from position 2 to position 4, and everything shifted.

This is going to cause problems with `.union()` because it matches columns **by position**, not by name!

### Comparing the Two DataFrames

Let's see both DataFrames side by side to understand the schema mismatch:
* `my_df1`: id, name, salary, mngr_id
* `wrong_df`: id, salary, mngr_id, name

Same data, same column names, but completely different order. This is a common real-world scenario when combining data from different sources.

### Why .union() Fails with Shuffled Columns

**This will error!** Here's why:

`.union()` matches columns **by position**, not by name:
* Position 1: id (bigint) with id (bigint) ✓
* Position 2: name (string) with salary (bigint) ✗ - Tries to cast 'Sohan' to bigint!
* Position 3: salary (bigint) with mngr_id (bigint) ✓
* Position 4: mngr_id (bigint) with name (string) ✗ - Tries to cast string to bigint!

Spark tries to cast the string 'Sohan' to a bigint and fails with `CAST_INVALID_INPUT`.

**Key lesson:** Column names don't matter to `.union()` - only position and type compatibility matter!

### Solution: unionByName() to the Rescue!

`unionByName()` matches columns **by name** instead of position, so it handles shuffled columns correctly.

**Answering your question:** "Why is this used for?"
* Use `unionByName()` when you're not 100% sure columns are in the same order
* It's safer and more explicit about matching logic

**About the name matching:** You're right that column names must match exactly! If one DataFrame has `name` and another has `nam`, it will error. But if names match, the order doesn't matter - `unionByName()` will align them correctly.

This is much more reliable than `.union()` when working with DataFrames from different sources.

### ⚠️ Problem 2: What Happens When Column Counts Differ?

Now we're creating a DataFrame with **more columns** than `my_df1`:
* `my_df1` has: 4 columns `["id", "name", "salary", "mngr_id"]`
* `wrong_df2` has: 5 columns `["id", "name", "salary", "mngr_id", "bonus"]`

Extra column alert! Let's see what happens when we try to union these.

### Why Union Fails with Different Column Counts

**This used to error** with `NUM_COLUMNS_MISMATCH` because:
* `my_df1` has 4 columns
* `wrong_df2` has 5 columns

Spark's `.union()` requires the same number of columns in both DataFrames.

**The fix applied here:** We're now selecting only the 4 matching columns from `wrong_df2` before the union, so both DataFrames have the same structure.

### Handling Missing Columns with unionByName()

**What this does:** `unionByName()` with `allowMissingColumns=True` lets you union DataFrames with different column counts!

* Common columns get matched by name
* Missing columns get filled with `null`

Result:
* Rows from `my_df1` have `null` in the `bonus` column (they don't have it)
* Rows from `wrong_df2` have their actual `bonus` values

This is super useful when combining data from different time periods or sources where the schema evolved over time.

### Alternative Solution: Select Matching Columns

**Answering your question:** "Why did the first statement not run but the second did?"

Both statements actually work! Here's what's happening:

**First statement:** `my_df1.select(...).union(wrong_df2)`
* Selects 4 columns from `my_df1`
* `wrong_df2` has 5 columns
* **Works because** after the fix in Cell 15, we're selecting matching columns

**Second statement:** `wrong_df2.select(...).union(my_df1)`
* Selects 4 columns from `wrong_df2` (dropping `bonus`)
* `my_df1` naturally has 4 columns
* Both have same column count → works!

**The pattern:** When unioning DataFrames with different columns, select only the columns you need from both sides before the union. This gives you control over which columns to keep.

In [0]:
my_df1.show()
wrong_df.show()

+---+-----+------+-------+
| id| name|salary|mngr_id|
+---+-----+------+-------+
| 19|Sohan| 50000|     18|
| 20| Sima| 75000|     17|
+---+-----+------+-------+

+---+------+-------+-----+
| id|salary|mngr_id| name|
+---+------+-------+-----+
| 19| 50000|     18|Sohan|
| 20| 75000|     17| Sima|
+---+------+-------+-----+



In [0]:
#explaining why it will fail 
my_df1.union(wrong_df).show()

---------------------------------------------------------------------------
NumberFormatException                     Traceback (most recent call last)
File <command-8479198741695862>, line 1
----> 1 my_df1.union(wrong_df).show()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:1156, in DataFrame.show(self, n, truncate, vertical)
   1155 def show(self, n: int = 20, truncate: Union[bool, int] = True, vertical: bool = False) -> None:
-> 1156     print(self._show_string(n, truncate, vertical))

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:909, in DataFrame._show_string(self, n, truncate, vertical)
    892     except ValueError:
    893         raise PySparkTypeError(
    894             errorClass="NOT_BOOL",
    895             messageParameters={
   (...)
    898             },
    899         )
    901 table, _ = DataFrame(
    902     plan.ShowString(
    903         child=self._plan,
    904         num_ro

In [0]:
# why is this used for , but this will only work if the names are there if one df : name and  df2 :nam then it wont work and give error 
my_df1.unionByName(wrong_df).show()

+---+-----+------+-------+
| id| name|salary|mngr_id|
+---+-----+------+-------+
| 19|Sohan| 50000|     18|
| 20| Sima| 75000|     17|
| 19|Sohan| 50000|     18|
| 20| Sima| 75000|     17|
+---+-----+------+-------+



In [0]:
#when no of  columns are more : 
wrong_column_data2=[(19 ,'Sohan',50000, 18,600),
(20 ,'Sima',75000,  17,800)]

w_schema2 = ["id","name","salary","mngr_id","bonus"]

wrong_df2 = spark.createDataFrame(wrong_column_data2, w_schema2)

In [0]:
#why this happend explanation 
my_df1.union(wrong_df2).show()

+---+-----+------+-------+
| id| name|salary|mngr_id|
+---+-----+------+-------+
| 19|Sohan| 50000|     18|
| 20| Sima| 75000|     17|
| 19|Sohan| 50000|     18|
| 20| Sima| 75000|     17|
+---+-----+------+-------+



In [0]:
#what it does 
my_df1.unionByName(wrong_df2, allowMissingColumns= True).show()

+---+-----+------+-------+-----+
| id| name|salary|mngr_id|bonus|
+---+-----+------+-------+-----+
| 19|Sohan| 50000|     18| NULL|
| 20| Sima| 75000|     17| NULL|
| 19|Sohan| 50000|     18|  600|
| 20| Sima| 75000|     17|  800|
+---+-----+------+-------+-----+



In [0]:
#seleccting only required columns , also explain why first statment didnt ran buut the second did 
my_df1.select("id","name","salary","mngr_id").union(wrong_df2).show()
wrong_df2.select("id","name","salary","mngr_id").union(my_df1).show()


+---+-----+------+-------+
| id| name|salary|mngr_id|
+---+-----+------+-------+
| 19|Sohan| 50000|     18|
| 20| Sima| 75000|     17|
| 19|Sohan| 50000|     18|
| 20| Sima| 75000|     17|
+---+-----+------+-------+



## 🎯 Summary & Decision Guide

### Quick Decision Tree: Which Union Method Should I Use?

```
Are you writing Python/PySpark or SQL?
├─ SQL?
│   ├─ Want to remove duplicates? → Use UNION
│   └─ Want to keep duplicates? → Use UNION ALL (faster)
│
└─ PySpark?
    ├─ Columns in exact same order & same count?
    │   └─ df1.union(df2)
    │
    ├─ Columns might be in different order?
    │   └─ df1.unionByName(df2)
    │
    └─ Different number of columns?
        └─ Option 1: df1.unionByName(df2, allowMissingColumns=True)
        └─ Option 2: df1.select(...).union(df2.select(...))
```

### Key Takeaways

1. **PySpark `.union()` keeps duplicates** - It's equivalent to SQL's `UNION ALL`, not SQL's `UNION`

2. **Column matching is positional in `.union()`** - The most common mistake! If columns are in different order, you'll get wrong data or cast errors

3. **`unionByName()` is safer** - It matches by column name, so order doesn't matter. Use it by default unless you're 100% sure about column order

4. **Schema compatibility matters:**
   * Same column count: Required for `.union()`
   * Same column names: Required for `unionByName()`
   * Same types at same position: Required for `.union()`

5. **SQL UNION removes duplicates, UNION ALL keeps them** - UNION is slower because it has to check for duplicates

### Common Pitfalls to Avoid

❌ **Assuming `.union()` removes duplicates** - It doesn't! Use `.union().distinct()` if you need that

❌ **Ignoring column order** - Can lead to silent data corruption where values end up in wrong columns

❌ **Not checking schemas first** - Always `.printSchema()` or `.show()` both DataFrames before union

❌ **Using deprecated `.unionAll()`** - Just use `.union()` instead

### Real-World Use Cases

**When you'd use union operations:**
* Combining data from multiple time periods (this month + last month)
* Merging data from different regions/sources
* Appending new records to historical data
* Combining train/test datasets
* Concatenating results from parallel processing

**Pro tip:** In production pipelines, always use `unionByName()` for safety - it's more maintainable when schemas evolve over time!